# Amazon Bedrock AgentCore Runtime의 TypeScript MCP Server에서 MCP Client 테스트

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime 환경을 사용하여 TypeScript 기반 MCP(Model Context Protocol) server를 호스팅하는 방법을 알아봅니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | TypeScript MCP server 호스팅                              |
| Tool 유형           | MCP server                                                |
| 튜토리얼 구성 요소  | AgentCore Runtime에 TypeScript MCP server 호스팅         |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 쉬움                                                       |
| 사용 SDK            | Anthropic TypeScript SDK for MCP                          |


### 튜토리얼 개요

1. AgentCore Runtime 인증은 Amazon Cognito를 사용하여 배포된 MCP server에 액세스할 JWT token을 제공합니다.

2. MCP server는 TypeScript로 작성되며 [custom flow를 사용하여 배포](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html)합니다.

3. MCP client는 Python으로 작성됩니다.
   _MCP client는 어떤 언어로도 작성할 수 있습니다._

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
- Node.js v22 이상(MCP server)
- Python 3.10+(MCP client)
- Docker(containerization용)  
- Docker image를 저장할 Amazon ECR(Elastic Container Registry)  
- Bedrock AgentCore 액세스 권한이 있는 AWS 계정  
- MCP (Model Context Protocol) library
- 실행 중인 Docker

In [ ]:
#!uv add -r requirements.txt --active

## MCP(Model Context Protocol) 이해

MCP는 AI 모델이 외부 데이터와 tool에 안전하게 액세스하도록 지원하는 protocol입니다. 주요 개념은 다음과 같습니다.

* **Tools**: AI가 작업을 수행하기 위해 호출할 수 있는 함수
* **Prompts**: server가 LLM과 상호 작용하기 위한 structured message와 지침을 제공하도록 지원
* **Streamable HTTP**: AgentCore Runtime에서 사용하는 transport protocol
* **Session Isolation**: 각 client가 `Mcp-Session-Id` header를 통해 격리된 session을 사용
* **Stateless Operation**: 확장성을 위해 server가 stateless operation을 지원해야 함

AgentCore Runtime은 MCP server가 기본 path인 `0.0.0.0:8000/mcp`에 호스팅되기를 기대합니다.

## 1단계: 인증을 위한 Amazon Cognito 설정

AgentCore Runtime에는 인증이 필요합니다. Amazon Cognito를 사용하여 배포된 MCP server에 액세스할 JWT token을 제공합니다.

In [ ]:
import sys
import os

# 현재 Notebook 디렉터리 가져오기
current_dir = os.path.dirname(os.path.abspath("__file__" if "__file__" in globals() else "."))

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# sys.path에 추가
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role, setup_cognito_user_pool

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## 2단계: IAM Execution Role 생성

시작하기 전에 AgentCore Runtime용 IAM role을 생성합니다. 이 role은 Runtime 실행에 필요한 권한을 제공합니다.

In [ ]:
tool_name = "mcp_server_ac"
print(f"Creating IAM role for {tool_name}...")
agentcore_iam_role = create_agentcore_role(agent_name=tool_name)
print("IAM role created ✓")
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")

## 3단계: MCP Server 생성

간단한 tool 두 개와 prompt 하나가 포함된 TypeScript MCP server를 생성합니다. 이 튜토리얼 아래의 src 폴더로 이동합니다.

1. dependency 설치

```
npm install
```

2. AWS credentials 설정
```
aws configure
export AWS_ACCESS_KEY_ID=your_access_key
export AWS_SECRET_ACCESS_KEY=your_secret_key
export AWS_REGION=us-east-1
```

3. server 시작(로컬 실행)
```
npm run start
```

## 4단계: Docker를 통한 MCP Server 배포

참고: 다음은 starter toolkit 없이 에이전트 또는 MCP server를 배포하는 수동 단계입니다. 

https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

1. ECR Repository 생성
```
aws ecr create-repository --repository-name mcp-server --region us-east-1
```
2. Image를 빌드하여 ECR에 push
```
# login token 가져오기
aws ecr get-login-password --region us-east-1 | \
  docker login --username AWS --password-stdin [account-id].dkr.ecr.us-east-1.amazonaws.com

docker buildx --platform linux/arm64 \
  -t [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest --push .
```

3. Bedrock AgentCore에 배포

    - AWS Console → Bedrock → AgentCore → Create Agent로 이동
    - protocol로 MCP 선택
    - Agent Runtime 구성:
        - Image URI: [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest
        - Bedrock 모델 액세스를 위한 IAM Permissions 설정
        - Agent Sandbox에서 배포 및 테스트
    - Discovery URL: 위의 cognito_config['discovery_url'] URL 선택
    - Client ID: 위의 cognito_config['client_id'] client ID 선택
    - execution role: 위의 agentcore_iam_role['Role']['Arn'] ARN 선택


## 5단계: Remote Access용 구성 저장

배포된 MCP server를 호출하기 전에 쉽게 가져올 수 있도록 Agent ARN(4단계에서 가져옴)과 Cognito 구성을 AWS Systems Manager Parameter Store 및 AWS Secrets Manager에 저장합니다.

In [ ]:
import boto3
import json

boto_session = Session()
region = boto_session.region_name

ssm_client = boto3.client("ssm", region_name=region)
secrets_client = boto3.client("secretsmanager", region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name="mcp_server/cognito/credentials",
        Description="Cognito credentials for MCP server",
        SecretString=json.dumps(cognito_config),
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId="mcp_server/cognito/credentials",
        SecretString=json.dumps(cognito_config),
    )
    print("✓ Cognito credentials updated in Secrets Manager")

# 참고: 4단계에서 생성한 agent ARN 추가
agent_arn_response = ssm_client.put_parameter(
    Name="/mcp_server/runtime/agent_arn",
    Value="Add your agent arn that you created in step 4",
    Type="String",
    Description="Agent ARN for MCP server",
    Overwrite=True,
)
print("✓ Agent ARN stored in Parameter Store")

## 6단계: Remote Testing Client 생성

배포된 MCP server를 테스트할 client를 생성합니다. 이 client는 AWS에서 필요한 credentials를 가져와 배포된 server에 연결합니다.

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")
     
        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    

    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## 7단계: 배포된 MCP Server 테스트

remote client를 사용하여 배포된 MCP server를 테스트합니다.

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

## 8단계: MCP Tool 원격 호출

tool 목록을 표시할 뿐 아니라 직접 호출하여 전체 MCP 기능을 보여주는 향상된 client를 생성합니다.

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)
                
                try:
                    print("\n➕ Testing add(5, 3)...")
                    add_result = await session.call_tool(
                        name="add",
                        arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                try:
                    print("\n✖️  Testing subtract(10, 2)...")
                    substract_result = await session.call_tool(
                        name="subtract",
                        arguments={"a": 10, "b": 2}
                    )
                    print(f"   Result: {substract_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                print("\n✅ MCP tool testing completed!")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Tool 호출 테스트

MCP tool을 직접 호출하여 테스트합니다.

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

## 다음 단계

MCP server를 AgentCore Runtime에 성공적으로 배포했으므로 다음 작업을 수행할 수 있습니다.

1. **Tool 추가**: MCP server에 추가 tool 확장
2. **Custom 인증**: custom JWT authorizer 구현
3. **통합**: 다른 AgentCore service와 통합

# 🎉 축하합니다!

다음 작업을 성공적으로 완료했습니다.

✅ custom tool이 포함된 **TypeScript MCP server 생성**  
✅ Amazon Cognito로 **인증 설정**  
✅ AgentCore Runtime을 사용하여 **AWS에 배포**  
✅ 적절한 인증으로 **원격 호출**  
✅ **MCP 개념과 best practice 학습**  

이제 MCP server가 Amazon Bedrock AgentCore Runtime에서 실행되고 있으며 production에서 사용할 준비가 되었습니다.